## Etapa 2: Análise Exploratória e Visualizações

In [ ]:
# ===================================================================
# PROJETO: Spotify Weekly Top 200 - Análise Exploratória das Américas
# OBJETIVO: Análise visual e insights das tendências musicais no continente americano
# CONTEXTO: Dados filtrados para países da América do Norte, Central e Sul
# ===================================================================

print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║                                                                              ║
║   🌎 SPOTIFY WEEKLY TOP 200 - ANÁLISE EXPLORATÓRIA DAS AMÉRICAS 🌎           ║
║                                                                              ║
║   Visualizações interativas e insights sobre tendências musicais             ║
║   no continente americano (Norte, Central e Sul)                             ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
""")

# ===================================================================
# CONFIGURAÇÕES INICIAIS
# ===================================================================

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Configurações de visualização
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

# Cores personalizadas para as Américas
CORES_AMERICAS = {
    'América do Norte': '#4ECDC4',
    'América Central': '#FFE66D',
    'Caribe': '#FF6B6B',
    'América do Sul': '#45B7D1',
    'Brasil': '#FF9F1C',
    'EUA': '#2EC4B6',
    'México': '#FFB703',
    'Argentina': '#9B5DE5'
}

print("✅ Bibliotecas importadas com sucesso!")
print(f"   🌎 Modo: Análise focada nas Américas")

# ===================================================================
# VERIFICAÇÃO DOS DADOS CARREGADOS
# ===================================================================

print("\n" + "="*70)
print("🔍 VERIFICANDO DADOS DAS AMÉRICAS")
print("="*70)

# Verificar se o DataFrame df existe
if 'df' not in dir():
    print("❌ DataFrame 'df' não encontrado!")
    print("   Execute primeiro o código de carregamento dos dados das Américas.")
    raise NameError("DataFrame 'df' não definido")

# Verificar se o DataFrame está vazio
if len(df) == 0:
    print("❌ DataFrame está vazio! Carregue os dados primeiro.")
    raise ValueError("DataFrame vazio")

# Estatísticas rápidas
print(f"\n✅ Dados carregados com sucesso!")
print(f"   📊 Shape: {df.shape[0]:,} registros × {df.shape[1]} colunas")
print(f"   🌍 Países: {df['country'].nunique()}")
print(f"   🎵 Músicas únicas: {df['track_name'].nunique():,}")
print(f"   🎤 Artistas únicos: {df['artist_names'].nunique():,}")

# Verificar colunas disponíveis
colunas_disponiveis = df.columns.tolist()
print(f"\n📋 Colunas disponíveis para análise: {len(colunas_disponiveis)}")

# Verificar se as colunas essenciais existem
colunas_essenciais = ['streams_millions', 'danceability', 'energy', 'valence', 'country']
colunas_faltando = [col for col in colunas_essenciais if col not in colunas_disponiveis]

if colunas_faltando:
    print(f"⚠️ Colunas faltando: {colunas_faltando}")
    print("   Algumas visualizações podem ser limitadas.")

# ===================================================================
# INFORMAÇÕES GERAIS DO DATASET DAS AMÉRICAS
# ===================================================================

print("\n" + "="*70)
print("📊 INFORMAÇÕES GERAIS - AMÉRICAS")
print("="*70)

# Métricas principais
if 'streams_millions' in df.columns:
    print(f"\n💰 STREAMS TOTAIS NAS AMÉRICAS:")
    print(f"   • Total: {df['streams_millions'].sum():.1f} milhões")
    print(f"   • Média por música: {df['streams_millions'].mean():.2f}M")
    print(f"   • Música com mais streams: {df.loc[df['streams_millions'].idxmax(), 'track_name'][:50]}...")

if 'sub_region' in df.columns:
    print(f"\n🗺️ DISTRIBUIÇÃO POR SUB-REGIÃO:")
    for regiao, qtd in df['sub_region'].value_counts().items():
        pct = qtd / len(df) * 100
        barra = '█' * int(pct / 2)
        print(f"   • {regiao:20} : {qtd:8,} ({pct:.1f}%) {barra}")

if 'music_profile' in df.columns:
    print(f"\n🎭 PERFIS MUSICAIS DOMINANTES:")
    for perfil, qtd in df['music_profile'].value_counts().head(5).items():
        pct = qtd / len(df) * 100
        print(f"   • {perfil:25} : {pct:.1f}%")

# ===================================================================
# PRONTIDÃO PARA VISUALIZAÇÕES
# ===================================================================

print("\n" + "="*70)
print("🎨 PREPARANDO VISUALIZAÇÕES")
print("="*70)

print("""
📈 GRÁFICOS QUE SERÃO GERADOS:
   1. 🗺️ Mapa interativo das Américas
   2. 📊 Top 10 países por streams
   3. 🎵 Top 10 músicas nas Américas
   4. 💃 Perfil musical por sub-região (Radar)
   5. 📈 Evolução temporal (Top 5 países)
   6. 🔥 Matriz de correlação musical
   7. 🎭 Distribuição de perfis musicais
   8. 📊 Dashboard integrado com 4 gráficos
""")

# Verificar se há dados suficientes para os gráficos
if len(df) < 100:
    print("⚠️ ATENÇÃO: Poucos dados para gerar gráficos significativos!")
    print(f"   Apenas {len(df)} registros. Recomenda-se pelo menos 1000 registros.")

print("\n✅ Configuração concluída! Pronto para gerar as visualizações.")
print("="*70)

# ===================================================================
# FUNÇÕES AUXILIARES PARA VISUALIZAÇÕES
# ===================================================================

def safe_sample(df, n=1000):
    """Amostragem segura para gráficos (evita erro se df for pequeno)"""
    return df.sample(min(n, len(df)))

def get_top_paises(df, n=10):
    """Retorna top N países por streams"""
    return (df.groupby('country')['streams_millions']
            .sum()
            .sort_values(ascending=False)
            .head(n)
            .reset_index())

def get_top_musicas(df, n=10):
    """Retorna top N músicas por streams"""
    return (df.groupby(['track_name', 'artist_names'])['streams_millions']
            .sum()
            .sort_values(ascending=False)
            .head(n)
            .reset_index())

print("✅ Funções auxiliares carregadas!")
print("\n🎯 Pronto para gerar os gráficos interativos!")


╔══════════════════════════════════════════════════════════════════════════════╗
║                                                                              ║
║   🌎 SPOTIFY WEEKLY TOP 200 - ANÁLISE EXPLORATÓRIA DAS AMÉRICAS 🌎           ║
║                                                                              ║
║   Visualizações interativas e insights sobre tendências musicais             ║
║   no continente americano (Norte, Central e Sul)                             ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝

✅ Bibliotecas importadas com sucesso!
   🌎 Modo: Análise focada nas Américas

🔍 VERIFICANDO DADOS DAS AMÉRICAS

✅ Dados carregados com sucesso!
   📊 Shape: 245,338 registros × 26 colunas
   🌍 Países: 17
   🎵 Músicas únicas: 5,479
   🎤 Artistas únicos: 3,481

📋 Colunas disponíveis para análise: 26

📊 INFORMAÇÕES GERAIS - AMÉRICAS

💰 STREAMS TOTAIS NAS

In [ ]:
# ===================================================================
# CARREGAR DADOS LIMPOS DAS AMÉRICAS
# ===================================================================
# Explicação: Carregar os dados previamente salvos e filtrados para as Américas

print("📂 CARREGANDO DADOS LIMPOS DAS AMÉRICAS...")
print("=" * 70)

# Caminhos dos arquivos (priorizar a pasta específica das Américas)
CAMINHOS = [
    '/content/drive/MyDrive/spotify_americas_results/spotify_americas_clean.parquet',
    '/content/drive/MyDrive/spotify_americas_clean.parquet',
    '/content/drive/MyDrive/spotify_americas_results/spotify_americas_clean.csv',
    '/content/drive/MyDrive/spotify_americas_clean.csv',
    '/content/drive/MyDrive/spotify_data_clean.parquet',  # fallback
    '/content/drive/MyDrive/spotify_data_clean.csv'       # fallback
]

df = None
caminho_usado = None

# Tentar carregar de cada caminho
for caminho in CAMINHOS:
    try:
        if caminho.endswith('.parquet'):
            df = pd.read_parquet(caminho)
            print(f"✅ Dados carregados do arquivo PARQUET!")
            print(f"   📁 Caminho: {caminho}")
            caminho_usado = caminho
            break
        elif caminho.endswith('.csv'):
            df = pd.read_csv(caminho)
            print(f"✅ Dados carregados do arquivo CSV!")
            print(f"   📁 Caminho: {caminho}")
            caminho_usado = caminho
            break
    except Exception as e:
        continue

# Se não encontrou nenhum arquivo
if df is None:
    print("❌ NENHUM ARQUIVO ENCONTRADO!")
    print("\n📁 Arquivos disponíveis no Drive:")
    !ls -la /content/drive/MyDrive/*.parquet 2>/dev/null
    !ls -la /content/drive/MyDrive/*.csv 2>/dev/null
    !ls -la /content/drive/MyDrive/spotify_americas_results/ 2>/dev/null
    raise FileNotFoundError("Arquivo de dados limpos não encontrado! Execute o ETL primeiro.")

# ===================================================================
# VERIFICAÇÃO E VALIDAÇÃO DOS DADOS CARREGADOS
# ===================================================================

print("\n" + "="*70)
print("🔍 VERIFICANDO DADOS CARREGADOS")
print("="*70)

print(f"\n📊 Shape do dataset: {df.shape[0]:,} linhas × {df.shape[1]} colunas")
print(f"💾 Memória utilizada: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Verificar período
if 'week' in df.columns:
    df['week'] = pd.to_datetime(df['week'], errors='coerce')
    print(f"📅 Período: {df['week'].min().date()} até {df['week'].max().date()}")
    dias_total = (df['week'].max() - df['week'].min()).days
    print(f"   Duração: {dias_total} dias ({dias_total/7:.1f} semanas)")

# Verificar países
if 'country' in df.columns:
    print(f"🌍 Países nas Américas: {df['country'].nunique()}")
    print(f"   Lista: {', '.join(sorted(df['country'].unique())[:10])}{'...' if df['country'].nunique() > 10 else ''}")

# Verificar músicas e artistas
if 'track_name' in df.columns:
    print(f"🎵 Músicas únicas: {df['track_name'].nunique():,}")
if 'artist_names' in df.columns:
    print(f"🎤 Artistas únicos: {df['artist_names'].nunique():,}")

# Verificar colunas especiais das Américas
colunas_especiais = ['sub_region', 'music_profile', 'is_latin_style', 'success_score']
print(f"\n📋 COLUNAS ESPECIAIS DAS AMÉRICAS:")
for col in colunas_especiais:
    if col in df.columns:
        print(f"   ✅ {col}: presente")
    else:
        print(f"   ❌ {col}: não encontrada")

# ===================================================================
# AMOSTRA DOS DADOS
# ===================================================================

print("\n" + "="*70)
print("📋 AMOSTRA DOS DADOS CARREGADOS")
print("="*70)

# Selecionar colunas para mostrar
colunas_mostrar = ['track_name', 'artist_names', 'country', 'rank', 'streams_millions',
                   'danceability', 'energy', 'valence', 'sub_region', 'music_profile']

colunas_existentes = [col for col in colunas_mostrar if col in df.columns]

if colunas_existentes:
    print("\n📊 Primeiras 8 linhas:")
    print(df[colunas_existentes].head(8).to_string())

    print("\n\n📊 Amostra aleatória (5 linhas):")
    print(df[colunas_existentes].sample(min(5, len(df))).to_string())
else:
    print(df.head())

# ===================================================================
# ESTATÍSTICAS RÁPIDAS DAS AMÉRICAS
# ===================================================================

print("\n" + "="*70)
print("📊 ESTATÍSTICAS RÁPIDAS DAS AMÉRICAS")
print("="*70)

if 'streams_millions' in df.columns:
    print(f"\n💰 STREAMS:")
    print(f"   Total: {df['streams_millions'].sum():.1f} milhões")
    print(f"   Média por música: {df['streams_millions'].mean():.2f}M")
    print(f"   Mediana: {df['streams_millions'].median():.2f}M")
    print(f"   Mínimo: {df['streams_millions'].min():.2f}M")
    print(f"   Máximo: {df['streams_millions'].max():.2f}M")

if all(col in df.columns for col in ['danceability', 'energy', 'valence']):
    print(f"\n🎵 CARACTERÍSTICAS MUSICAIS (médias):")
    print(f"   Dançabilidade: {df['danceability'].mean():.3f}")
    print(f"   Energia: {df['energy'].mean():.3f}")
    print(f"   Positividade (Valence): {df['valence'].mean():.3f}")

if 'sub_region' in df.columns:
    print(f"\n🗺️ DISTRIBUIÇÃO POR SUB-REGIÃO:")
    for regiao, qtd in df['sub_region'].value_counts().items():
        pct = qtd / len(df) * 100
        print(f"   {regiao:20} : {qtd:8,} ({pct:.1f}%)")

if 'is_latin_style' in df.columns:
    pct_latin = df['is_latin_style'].mean() * 100
    print(f"\n🎵 ESTILO LATINO:")
    print(f"   Músicas com características latinas: {pct_latin:.1f}%")

# ===================================================================
# VERIFICAÇÃO DE QUALIDADE
# ===================================================================

print("\n" + "="*70)
print("🔍 VERIFICAÇÃO DE QUALIDADE DOS DADOS")
print("="*70)

# Verificar valores nulos
nulos = df.isnull().sum()
nulos = nulos[nulos > 0]
if len(nulos) > 0:
    print(f"\n⚠️ VALORES NULOS ENCONTRADOS:")
    for col, qtd in nulos.items():
        print(f"   • {col}: {qtd} ({qtd/len(df)*100:.2f}%)")
else:
    print("\n✅ Nenhum valor nulo encontrado!")

# Verificar duplicatas
duplicatas = df.duplicated().sum()
print(f"\n📋 DUPLICATAS: {duplicatas:,} registros duplicados")

# Verificar valores inconsistentes
if 'rank' in df.columns:
    ranks_invalidos = ((df['rank'] < 1) | (df['rank'] > 200)).sum()
    print(f"🎯 RANKS INVÁLIDOS: {ranks_invalidos}")

if 'streams' in df.columns:
    streams_zero = (df['streams'] <= 0).sum()
    print(f"🎯 STREAMS ZERO/NEGATIVO: {streams_zero}")

# ===================================================================
# RESUMO FINAL DO CARREGAMENTO
# ===================================================================

print("\n" + "="*70)
print("✅ CARREGAMENTO CONCLUÍDO COM SUCESSO!")
print("="*70)

print(f"""
📊 RESUMO DO DATASET DAS AMÉRICAS:
   • Total de registros: {df.shape[0]:,}
   • Total de colunas: {df.shape[1]}
   • Países representados: {df['country'].nunique() if 'country' in df.columns else 'N/A'}
   • Período: {df['week'].min().date() if 'week' in df.columns else 'N/A'} a {df['week'].max().date() if 'week' in df.columns else 'N/A'}
   • Memória: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB

📂 Origem: {caminho_usado}

🚀 Pronto para iniciar as visualizações interativas!
""")

print("="*70)

📂 CARREGANDO DADOS LIMPOS DAS AMÉRICAS...
✅ Dados carregados do arquivo PARQUET!
   📁 Caminho: /content/drive/MyDrive/spotify_americas_results/spotify_americas_clean.parquet

🔍 VERIFICANDO DADOS CARREGADOS

📊 Shape do dataset: 245,338 linhas × 26 colunas
💾 Memória utilizada: 204.62 MB
📅 Período: 2021-02-04 até 2022-07-14
   Duração: 525 dias (75.0 semanas)
🌍 Países nas Américas: 17
   Lista: Argentina, Bolivia, Brazil, Canada, Chile, Colombia, Costa Rica, Dominican Republic, Ecuador, Guatemala...
🎵 Músicas únicas: 5,479
🎤 Artistas únicos: 3,481

📋 COLUNAS ESPECIAIS DAS AMÉRICAS:
   ✅ sub_region: presente
   ✅ music_profile: presente
   ✅ is_latin_style: presente
   ✅ success_score: presente

📋 AMOSTRA DOS DADOS CARREGADOS

📊 Primeiras 8 linhas:
              track_name   artist_names    country  rank  streams_millions  danceability  energy  valence         sub_region                         music_profile
0                 Plan A   Paulo Londra  Argentina     1              3.00        